In [10]:
from pathlib import Path
import pandas as pd

raw_dir=Path("../data/raw")

pbp_2023_files=list(raw_dir.glob("*2023*.csv"))
pbp_2023_path=pbp_2023_files[0]

pbp_2023=pd.read_csv(pbp_2023_path)
print(pbp_2023.shape)
print(pbp_2023.columns)

(646367, 16)
Index(['gameid', 'period', 'clock', 'h_pts', 'a_pts', 'team', 'playerid',
       'player', 'type', 'subtype', 'result', 'x', 'y', 'dist', 'desc',
       'season'],
      dtype='object')


In [12]:
summary={}

for col in pbp_2023.columns:
    s=pbp_2023[col]
    info={
        "dtype":s.dtype,
        "n_unique":s.nunique(dropna=True)
    }

    if pd.api.types.is_numeric_dtype(s):
        info["min"]=s.min()
        info["max"]=s.max()
    else:
      info["sample_values"]=s.dropna().unique()[:10].tolist()

    summary[col]=info

summary

{'gameid': {'dtype': dtype('int64'),
  'n_unique': 1319,
  'min': np.int64(22200001),
  'max': np.int64(52200211)},
 'period': {'dtype': dtype('int64'),
  'n_unique': 6,
  'min': np.int64(1),
  'max': np.int64(6)},
 'clock': {'dtype': dtype('O'),
  'n_unique': 1261,
  'sample_values': ['PT12M00.00S',
   'PT11M38.00S',
   'PT11M35.00S',
   'PT11M15.00S',
   'PT11M05.00S',
   'PT11M03.00S',
   'PT10M46.00S',
   'PT10M33.00S',
   'PT10M31.00S',
   'PT10M25.00S']},
 'h_pts': {'dtype': dtype('float64'),
  'n_unique': 166,
  'min': np.float64(0.0),
  'max': np.float64(175.0)},
 'a_pts': {'dtype': dtype('float64'),
  'n_unique': 168,
  'min': np.float64(0.0),
  'max': np.float64(176.0)},
 'team': {'dtype': dtype('O'),
  'n_unique': 30,
  'sample_values': ['BOS',
   'PHI',
   'GSW',
   'LAL',
   'DET',
   'ORL',
   'IND',
   'WAS',
   'ATL',
   'HOU']},
 'playerid': {'dtype': dtype('int64'),
  'n_unique': 1259,
  'min': np.int64(0),
  'max': np.int64(1610612766)},
 'player': {'dtype': dtype('O

In [13]:
cols_to_inspect=["period","type","subtype","result","team"]

for col in cols_to_inspect:
    print("====",col,"====")
    print(pbp_2023[col].value_counts(dropna=False))
    print("\n")

==== period ====
period
4    163770
2    163490
3    157753
1    156400
5      4449
6       505
Name: count, dtype: int64


==== type ====
type
Rebound                                     135767
Missed Shot                                 122341
Made Shot                                   110394
Free Throw                                   61735
Substitution                                 61313
Foul                                         53948
Turnover                                     36939
NaN                                          31464
Timeout                                      14461
period                                       10736
Violation                                     2474
Jump Ball                                     2237
Instant Replay                                2105
Foul                                           223
Instant Replay                                 133
Ejection                                        88
Turnover                                

In [16]:
type_subtype_counts=(
        pbp_2023
        .groupby(["type","subtype"])
        .size()
        .reset_index(name="counts")
        .sort_values(by="counts", ascending=False)
)

type_subtype_counts.head(30)

,type,subtype,counts
142,Rebound,Unknown,129104
115,Missed Shot,Jump Shot,43175
16,Foul,Shooting,27669
20,Free Throw,Free Throw 1 of 2,25092
22,Free Throw,Free Throw 2 of 2,25076
67,Made Shot,Jump Shot,25053
117,Missed Shot,Pullup Jump shot,17716
14,Foul,Personal,15168
144,Timeout,Regular,14245
149,Turnover,Bad Pass,12429


In [23]:
type_subtype_counts[(type_subtype_counts["type"]=="Made Shot") | (type_subtype_counts["type"]=="Missed Shot")].head(10)

,type,subtype,counts
115,Missed Shot,Jump Shot,43175
67,Made Shot,Jump Shot,25053
117,Missed Shot,Pullup Jump shot,17716
69,Made Shot,Pullup Jump shot,11657
104,Missed Shot,Driving Layup Shot,10477
56,Made Shot,Driving Layup Shot,10031
132,Missed Shot,Step Back Jump shot,6940
102,Missed Shot,Driving Floating Jump Shot,6787
54,Made Shot,Driving Floating Jump Shot,5316
84,Made Shot,Step Back Jump shot,4394


In [34]:
import numpy as np
import pandas as pd

# 1) Partimos de las columnas básicas
states_2023 = pbp_2023[["gameid", "season", "period", "clock", "h_pts", "a_pts"]].copy()

# 2) Diferencia de puntos a favor del equipo local
states_2023["score_diff"] = states_2023["h_pts"] - states_2023["a_pts"]


def parse_clock_to_seconds(clock_str):
    """
    Convierte strings tipo 'PT12M00.00S' o 'PT11M38.00S' a segundos.
    """
    if pd.isna(clock_str):
        return np.nan
    try:
        s = str(clock_str).replace("PT", "")   # '12M00.00S'
        minutes_part, seconds_part = s.split("M")  # '12', '00.00S'
        seconds_part = seconds_part.rstrip("S")    # '00.00'
        seconds_float = float(seconds_part)
        return int(minutes_part) * 60 + int(seconds_float)
    except Exception:
        return np.nan

# 3) Segundos restantes en el PERIODO
states_2023["seconds_left_in_period"] = states_2023["clock"].apply(parse_clock_to_seconds)


def total_seconds_remaining(row):
    period = row["period"]
    sec_left = row["seconds_left_in_period"]

    base_game_seconds = 4 * 12 * 60  # 48 min = 2880 s

    if period <= 4:
        seconds_played_before = (period - 1) * 12 * 60
        period_length = 12 * 60
        total_game_seconds = base_game_seconds
    else:
        ot_number = period - 4
        seconds_played_before = base_game_seconds + (ot_number - 1) * 5 * 60
        period_length = 5 * 60
        total_game_seconds = base_game_seconds + ot_number * 5 * 60

    seconds_played_in_period = period_length - sec_left
    seconds_played = seconds_played_before + seconds_played_in_period

    return total_game_seconds - seconds_played

# 4) Segundos restantes en TODO el partido
states_2023["seconds_remaining"] = states_2023.apply(total_seconds_remaining, axis=1)

states_2023 = states_2023.dropna(subset=["score_diff"])

# 5) Verificamos
states_2023[["gameid", "period", "clock",
             "seconds_left_in_period", "seconds_remaining",
             "h_pts", "a_pts", "score_diff"]].head(10)


,gameid,period,clock,seconds_left_in_period,seconds_remaining,h_pts,a_pts,score_diff
0,22200001,1,PT12M00.00S,720,2880,0.0,0.0,0.0
6,22200001,1,PT11M15.00S,675,2835,2.0,0.0,2.0
9,22200001,1,PT11M03.00S,663,2823,2.0,2.0,0.0
10,22200001,1,PT10M46.00S,646,2806,5.0,2.0,3.0
14,22200001,1,PT10M25.00S,625,2785,6.0,2.0,4.0
15,22200001,1,PT10M25.00S,625,2785,7.0,2.0,5.0
29,22200001,1,PT09M42.00S,582,2742,9.0,2.0,7.0
42,22200001,1,PT08M49.00S,529,2689,9.0,3.0,6.0
46,22200001,1,PT08M24.00S,504,2664,9.0,3.0,6.0
47,22200001,1,PT08M24.00S,504,2664,9.0,4.0,5.0


In [35]:
# Marcador final por partido
final_score_2023 = (
    states_2023
    .groupby("gameid")[["h_pts", "a_pts"]]
    .last()
    .reset_index()
)

# 1 si el local ganó, 0 si no
final_score_2023["home_win"] = (final_score_2023["h_pts"] > final_score_2023["a_pts"]).astype(int)

final_score_2023.head()


,gameid,h_pts,a_pts,home_win
0,22200001,126.0,117.0,1
1,22200002,123.0,109.0,1
2,22200003,113.0,109.0,1
3,22200004,107.0,114.0,0
4,22200005,117.0,107.0,1


In [36]:
states_2023 = states_2023.merge(
    final_score_2023[["gameid", "home_win"]],
    on="gameid",
    how="left"
)

states_2023[["gameid", "period", "clock",
             "seconds_remaining", "score_diff", "home_win"]].head(10)


,gameid,period,clock,seconds_remaining,score_diff,home_win
0,22200001,1,PT12M00.00S,2880,0.0,1
1,22200001,1,PT11M15.00S,2835,2.0,1
2,22200001,1,PT11M03.00S,2823,0.0,1
3,22200001,1,PT10M46.00S,2806,3.0,1
4,22200001,1,PT10M25.00S,2785,4.0,1
5,22200001,1,PT10M25.00S,2785,5.0,1
6,22200001,1,PT09M42.00S,2742,7.0,1
7,22200001,1,PT08M49.00S,2689,6.0,1
8,22200001,1,PT08M24.00S,2664,6.0,1
9,22200001,1,PT08M24.00S,2664,5.0,1
